# Baseline Probability-of-Default Model

## Objective

This notebook develops an interpretable baseline logistic-regression
model for the validated 24-month credit outcome.

## Time-Based Modeling Design

- 2015 loans: initial model training;
- 2016 loans: internal later-vintage validation;
- 2015–2016 loans: final model development;
- 2017 loans: untouched final test population; and
- 2006 loans: historical stress comparison.

## Model Controls

All preprocessing and model estimation will be performed through one
pipeline.

Imputation, scaling, and categorical encoding will be fitted using only
the applicable training or development population. No information from
the 2017 test population will influence model fitting.

The baseline model will not use class weighting because its predicted
probabilities are intended to approximate observed probability of
default. Class imbalance will instead be addressed through appropriate
evaluation metrics.

## Primary Evaluation Metrics

Because defaults are rare, overall accuracy is not an appropriate
primary measure. Evaluation will emphasize:

- ROC AUC;
- precision-recall AUC;
- Brier score;
- calibration;
- default capture within high-risk segments; and
- performance stability across vintages.

In [2]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 53


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_directory = (
    project_root / "data" / "processed"
)

models_directory = (
    project_root / "models"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True,
)

modeling_dataset_path = (
    processed_data_directory
    / "modeling_dataset.parquet"
)

feature_manifest_path = (
    processed_data_directory
    / "feature_manifest.csv"
)


modeling_dataset = pd.read_parquet(
    modeling_dataset_path
)

feature_manifest = pd.read_csv(
    feature_manifest_path
)

numeric_features = (
    feature_manifest.loc[
        feature_manifest[
            "feature_type"
        ].eq("numeric"),
        "feature",
    ]
    .tolist()
)

categorical_features = (
    feature_manifest.loc[
        feature_manifest[
            "feature_type"
        ].eq("categorical"),
        "feature",
    ]
    .tolist()
)

predictor_features = (
    numeric_features
    + categorical_features
)


setup_check = pd.Series(
    {
        "scikit_learn_version": (
            sklearn.__version__
        ),
        "modeling_rows": len(
            modeling_dataset
        ),
        "modeling_columns": len(
            modeling_dataset.columns
        ),
        "numeric_features": len(
            numeric_features
        ),
        "categorical_features": len(
            categorical_features
        ),
        "total_predictors": len(
            predictor_features
        ),
        "modeling_file_exists": (
            modeling_dataset_path.exists()
        ),
        "manifest_file_exists": (
            feature_manifest_path.exists()
        ),
    },
    name="result",
)

display(setup_check)

scikit_learn_version     1.9.0
modeling_rows           200000
modeling_columns            30
numeric_features            15
categorical_features         9
total_predictors            24
modeling_file_exists      True
manifest_file_exists      True
Name: result, dtype: object

In [3]:
def prepare_population(
    data,
    vintages,
):
    population = data.loc[
        data["vintage"].isin(vintages)
        & data["eligible_24m"]
    ].copy()

    X = population[
        predictor_features
    ].copy()

    # Standard numeric representation.
    for feature in numeric_features:
        X[feature] = pd.to_numeric(
            X[feature],
            errors="coerce",
        ).astype("float64")

    # Convert categorical missing values to standard
    # NumPy missing values for scikit-learn.
    for feature in categorical_features:
        category_values = (
            X[feature]
            .astype("string")
            .astype("object")
        )

        category_values.loc[
            pd.isna(category_values)
        ] = np.nan

        X[feature] = category_values

    y = (
        population["default_24m"]
        .astype("int8")
    )

    identifiers = population[
        [
            "loan_identifier",
            "vintage",
        ]
    ].copy()

    return X, y, identifiers


X_train_2015, y_train_2015, ids_train_2015 = (
    prepare_population(
        modeling_dataset,
        [2015],
    )
)

X_validation_2016, y_validation_2016, ids_validation_2016 = (
    prepare_population(
        modeling_dataset,
        [2016],
    )
)

X_development, y_development, ids_development = (
    prepare_population(
        modeling_dataset,
        [2015, 2016],
    )
)

X_test_2017, y_test_2017, ids_test_2017 = (
    prepare_population(
        modeling_dataset,
        [2017],
    )
)

X_historical_2006, y_historical_2006, ids_historical_2006 = (
    prepare_population(
        modeling_dataset,
        [2006],
    )
)


population_summary = pd.DataFrame(
    [
        {
            "population": "initial_training",
            "vintages": "2015",
            "rows": len(y_train_2015),
            "defaults": int(
                y_train_2015.sum()
            ),
            "default_rate_percent": round(
                y_train_2015.mean() * 100,
                3,
            ),
        },
        {
            "population": "internal_validation",
            "vintages": "2016",
            "rows": len(
                y_validation_2016
            ),
            "defaults": int(
                y_validation_2016.sum()
            ),
            "default_rate_percent": round(
                y_validation_2016.mean()
                * 100,
                3,
            ),
        },
        {
            "population": "final_development",
            "vintages": "2015–2016",
            "rows": len(y_development),
            "defaults": int(
                y_development.sum()
            ),
            "default_rate_percent": round(
                y_development.mean() * 100,
                3,
            ),
        },
        {
            "population": "final_test",
            "vintages": "2017",
            "rows": len(y_test_2017),
            "defaults": int(
                y_test_2017.sum()
            ),
            "default_rate_percent": round(
                y_test_2017.mean() * 100,
                3,
            ),
        },
        {
            "population": "historical_comparison",
            "vintages": "2006",
            "rows": len(
                y_historical_2006
            ),
            "defaults": int(
                y_historical_2006.sum()
            ),
            "default_rate_percent": round(
                y_historical_2006.mean()
                * 100,
                3,
            ),
        },
    ]
)

population_control_check = pd.Series(
    {
        "missing_train_targets": int(
            y_train_2015.isna().sum()
        ),
        "missing_validation_targets": int(
            y_validation_2016.isna().sum()
        ),
        "missing_test_targets": int(
            y_test_2017.isna().sum()
        ),
        "train_validation_overlap": len(
            set(
                ids_train_2015[
                    "loan_identifier"
                ]
            )
            & set(
                ids_validation_2016[
                    "loan_identifier"
                ]
            )
        ),
        "development_test_overlap": len(
            set(
                ids_development[
                    "loan_identifier"
                ]
            )
            & set(
                ids_test_2017[
                    "loan_identifier"
                ]
            )
        ),
    },
    name="result",
)

display(population_summary)
display(population_control_check)

,population,vintages,rows,defaults,default_rate_percent
0,initial_training,2015,39911,253,0.634
1,internal_validation,2016,42476,330,0.777
2,final_development,2015–2016,82387,583,0.708
3,final_test,2017,42064,397,0.944
4,historical_comparison,2006,37555,967,2.575


missing_train_targets         0
missing_validation_targets    0
missing_test_targets          0
train_validation_overlap      0
development_test_overlap      0
Name: result, dtype: int64

In [4]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=True,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)

initial_pd_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)


initial_pd_pipeline.fit(
    X_train_2015,
    y_train_2015,
)


train_2015_probability = (
    initial_pd_pipeline.predict_proba(
        X_train_2015
    )[:, 1]
)

validation_2016_probability = (
    initial_pd_pipeline.predict_proba(
        X_validation_2016
    )[:, 1]
)


def evaluate_probability_model(
    y_true,
    probability,
    population_name,
):
    probability = np.asarray(
        probability
    )

    top_decile_cutoff = np.quantile(
        probability,
        0.90,
    )

    top_decile_mask = (
        probability >= top_decile_cutoff
    )

    observed_default_rate = (
        y_true.mean()
    )

    top_decile_default_rate = (
        y_true.loc[
            top_decile_mask
        ].mean()
    )

    defaults_captured = (
        y_true.loc[
            top_decile_mask
        ].sum()
        / y_true.sum()
    )

    return {
        "population": population_name,
        "rows": len(y_true),
        "defaults": int(
            y_true.sum()
        ),
        "observed_default_rate": (
            observed_default_rate
        ),
        "mean_predicted_probability": (
            probability.mean()
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probability,
        ),
        "average_precision": (
            average_precision_score(
                y_true,
                probability,
            )
        ),
        "brier_score": brier_score_loss(
            y_true,
            probability,
        ),
        "top_decile_default_rate": (
            top_decile_default_rate
        ),
        "top_decile_lift": (
            top_decile_default_rate
            / observed_default_rate
        ),
        "top_decile_default_capture": (
            defaults_captured
        ),
    }


initial_model_results = pd.DataFrame(
    [
        evaluate_probability_model(
            y_train_2015,
            train_2015_probability,
            "2015 training",
        ),
        evaluate_probability_model(
            y_validation_2016,
            validation_2016_probability,
            "2016 validation",
        ),
    ]
)


trained_preprocessor = (
    initial_pd_pipeline.named_steps[
        "preprocessor"
    ]
)

transformed_feature_names = (
    trained_preprocessor
    .get_feature_names_out()
)

model_fit_check = pd.Series(
    {
        "transformed_feature_count": len(
            transformed_feature_names
        ),
        "model_iterations": int(
            initial_pd_pipeline
            .named_steps["model"]
            .n_iter_[0]
        ),
        "maximum_iterations": 1_000,
        "model_converged": bool(
            initial_pd_pipeline
            .named_steps["model"]
            .n_iter_[0]
            < 1_000
        ),
    },
    name="result",
)

display(model_fit_check)
display(
    initial_model_results.style.format(
        {
            "observed_default_rate": "{:.3%}",
            "mean_predicted_probability": "{:.3%}",
            "roc_auc": "{:.3f}",
            "average_precision": "{:.3f}",
            "brier_score": "{:.5f}",
            "top_decile_default_rate": "{:.3%}",
            "top_decile_lift": "{:.2f}",
            "top_decile_default_capture": "{:.1%}",
        }
    )
)

transformed_feature_count      84
model_iterations               11
maximum_iterations           1000
model_converged              True
Name: result, dtype: object

,population,rows,defaults,observed_default_rate,mean_predicted_probability,roc_auc,average_precision,brier_score,top_decile_default_rate,top_decile_lift,top_decile_default_capture
0,2015 training,39911,253,0.634%,0.647%,0.841,0.042,0.00622,3.557%,5.61,56.1%
1,2016 validation,42476,330,0.777%,0.551%,0.843,0.045,0.00758,4.025%,5.18,51.8%


In [5]:
from sklearn.base import clone


# Create a fresh copy so the final model is fitted
# independently on the combined development population.
final_pd_pipeline = clone(
    initial_pd_pipeline
)

final_pd_pipeline.fit(
    X_development,
    y_development,
)


development_probability = (
    final_pd_pipeline.predict_proba(
        X_development
    )[:, 1]
)

test_2017_probability = (
    final_pd_pipeline.predict_proba(
        X_test_2017
    )[:, 1]
)

historical_2006_probability = (
    final_pd_pipeline.predict_proba(
        X_historical_2006
    )[:, 1]
)


final_model_results = pd.DataFrame(
    [
        evaluate_probability_model(
            y_development,
            development_probability,
            "2015–2016 development",
        ),
        evaluate_probability_model(
            y_test_2017,
            test_2017_probability,
            "2017 final test",
        ),
        evaluate_probability_model(
            y_historical_2006,
            historical_2006_probability,
            "2006 historical comparison",
        ),
    ]
)


final_model_fit_check = pd.Series(
    {
        "development_rows_used": len(
            y_development
        ),
        "development_defaults_used": int(
            y_development.sum()
        ),
        "test_rows_not_used_in_fit": len(
            y_test_2017
        ),
        "historical_rows_not_used_in_fit": len(
            y_historical_2006
        ),
        "model_iterations": int(
            final_pd_pipeline
            .named_steps["model"]
            .n_iter_[0]
        ),
        "model_converged": bool(
            final_pd_pipeline
            .named_steps["model"]
            .n_iter_[0]
            < 1_000
        ),
        "transformed_feature_count": len(
            final_pd_pipeline
            .named_steps["preprocessor"]
            .get_feature_names_out()
        ),
    },
    name="result",
)

display(final_model_fit_check)

display(
    final_model_results.style.format(
        {
            "observed_default_rate": "{:.3%}",
            "mean_predicted_probability": "{:.3%}",
            "roc_auc": "{:.3f}",
            "average_precision": "{:.3f}",
            "brier_score": "{:.5f}",
            "top_decile_default_rate": "{:.3%}",
            "top_decile_lift": "{:.2f}",
            "top_decile_default_capture": "{:.1%}",
        }
    )
)

development_rows_used              82387
development_defaults_used            583
test_rows_not_used_in_fit          42064
historical_rows_not_used_in_fit    37555
model_iterations                      10
model_converged                     True
transformed_feature_count             84
Name: result, dtype: object

,population,rows,defaults,observed_default_rate,mean_predicted_probability,roc_auc,average_precision,brier_score,top_decile_default_rate,top_decile_lift,top_decile_default_capture
0,2015–2016 development,82387,583,0.708%,0.714%,0.857,0.048,0.00691,3.993%,5.64,56.4%
1,2017 final test,42064,397,0.944%,0.879%,0.840,0.059,0.00912,4.754%,5.04,50.4%
2,2006 historical comparison,37555,967,2.575%,3.255%,0.802,0.091,0.02531,9.824%,3.82,38.2%


In [7]:
from pathlib import Path
import joblib

project_root = Path(
    r"C:\GitHub Projects\freddie-mac-credit-risk"
)

model_path = (
    project_root
    / "models"
    / "baseline_pd_logistic.joblib"
)

model_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    final_pd_pipeline,
    model_path
)

print("Model saved:", model_path.exists())
print("Location:", model_path)
print(
    "File size MB:",
    round(model_path.stat().st_size / (1024 ** 2), 2)
)

Model saved: True
Location: C:\GitHub Projects\freddie-mac-credit-risk\models\baseline_pd_logistic.joblib
File size MB: 0.01


In [1]:
from pathlib import Path
import joblib

project_root = Path(
    r"C:\GitHub Projects\freddie-mac-credit-risk"
)

model_path = (
    project_root
    / "models"
    / "baseline_pd_logistic.joblib"
)

reloaded_pd_pipeline = joblib.load(model_path)

print("Model file exists:", model_path.exists())
print("Model reloaded successfully: True")
print(
    "Pipeline steps:",
    list(reloaded_pd_pipeline.named_steps.keys())
)

Model file exists: True
Model reloaded successfully: True
Pipeline steps: ['preprocessor', 'model']


In [2]:
import pandas as pd

modeling_path = (
    project_root
    / "data"
    / "processed"
    / "modeling_dataset.parquet"
)

modeling_data = pd.read_parquet(modeling_path)

predictor_columns = list(
    reloaded_pd_pipeline
    .named_steps["preprocessor"]
    .feature_names_in_
)

non_predictor_columns = [
    column
    for column in modeling_data.columns
    if column not in predictor_columns
]

print("Modeling rows:", len(modeling_data))
print("Modeling columns:", modeling_data.shape[1])
print("Expected predictors:", len(predictor_columns))
print("Non-predictor columns:", non_predictor_columns)

Modeling rows: 200000
Modeling columns: 30
Expected predictors: 24
Non-predictor columns: ['loan_identifier', 'vintage', 'population_role', 'eligible_24m', 'censored_24m', 'default_24m']


In [3]:
import numpy as np

eligibility_mask = (
    modeling_data["eligible_24m"]
    .astype("boolean")
    .fillna(False)
)

X_scoring = modeling_data.loc[
    eligibility_mask,
    predictor_columns
].copy()

# Convert categorical missing values into a format
# that the saved scikit-learn pipeline can process.
categorical_columns = X_scoring.select_dtypes(
    include=["object", "string", "category"]
).columns

for column in categorical_columns:
    category_values = (
        X_scoring[column]
        .astype("string")
        .astype("object")
    )
    category_values.loc[pd.isna(category_values)] = np.nan
    X_scoring[column] = category_values

predicted_probability = (
    reloaded_pd_pipeline
    .predict_proba(X_scoring)[:, 1]
)

prediction_output = modeling_data.loc[
    eligibility_mask,
    [
        "loan_identifier",
        "vintage",
        "population_role",
        "default_24m"
    ]
].copy()

prediction_output[
    "predicted_default_probability"
] = predicted_probability

prediction_path = (
    project_root
    / "data"
    / "processed"
    / "pd_predictions.parquet"
)

prediction_output.to_parquet(
    prediction_path,
    index=False
)

print("Prediction file saved:", prediction_path.exists())
print("Prediction rows:", len(prediction_output))
print(
    "Unique loan IDs:",
    prediction_output["loan_identifier"].nunique()
)
print(
    "Missing probabilities:",
    prediction_output[
        "predicted_default_probability"
    ].isna().sum()
)
print(
    "Minimum probability:",
    round(predicted_probability.min(), 6)
)
print(
    "Maximum probability:",
    round(predicted_probability.max(), 6)
)

c:\GitHub Projects\freddie-mac-credit-risk\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Prediction file saved: True
Prediction rows: 162006
Unique loan IDs: 162006
Missing probabilities: 0
Minimum probability: 4.3e-05
Maximum probability: 0.991191


In [4]:
preprocessor = (
    reloaded_pd_pipeline
    .named_steps["preprocessor"]
)

for transformer_name, transformer, columns in (
    preprocessor.transformers_
):
    print("Transformer:", transformer_name)
    print("Columns:", list(columns))
    print("Object:", transformer)
    print()

Transformer: numeric
Columns: ['credit_score', 'mi_percentage', 'original_dti', 'original_upb_log', 'original_ltv_capped_200', 'original_cltv_capped_200', 'original_interest_rate', 'original_loan_term', 'number_of_borrowers', 'credit_score_missing', 'original_dti_missing', 'original_ltv_missing', 'original_cltv_missing', 'original_ltv_above_100', 'original_cltv_above_100']
Object: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

Transformer: categorical
Columns: ['first_time_homebuyer_indicator', 'number_of_units', 'occupancy_status', 'channel', 'property_state', 'property_type', 'loan_purpose', 'super_conforming_flag', 'harp_indicator']
Object: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(drop='first', handle_unknown='ignore'))])



In [5]:
categorical_pipeline = (
    preprocessor
    .named_transformers_["categorical"]
)

categorical_imputer = (
    categorical_pipeline
    .named_steps["imputer"]
)

categorical_encoder = (
    categorical_pipeline
    .named_steps["encoder"]
)

categorical_features = [
    "first_time_homebuyer_indicator",
    "number_of_units",
    "occupancy_status",
    "channel",
    "property_state",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
    "harp_indicator"
]

imputed_categories = pd.DataFrame(
    categorical_imputer.transform(
        X_scoring[categorical_features]
    ),
    columns=categorical_features,
    index=X_scoring.index
)

unknown_category_results = []

for position, feature in enumerate(
    categorical_features
):
    known_levels = set(
        categorical_encoder.categories_[position]
    )

    unknown_mask = ~imputed_categories[
        feature
    ].isin(known_levels)

    if unknown_mask.any():
        unknown_index = imputed_categories.index[
            unknown_mask
        ]

        detail = pd.DataFrame({
            "feature": feature,
            "unknown_level": (
                imputed_categories.loc[
                    unknown_index,
                    feature
                ].astype(str).to_numpy()
            ),
            "vintage": (
                modeling_data.loc[
                    unknown_index,
                    "vintage"
                ].to_numpy()
            ),
            "population_role": (
                modeling_data.loc[
                    unknown_index,
                    "population_role"
                ].to_numpy()
            )
        })

        unknown_category_results.append(
            detail.groupby(
                [
                    "feature",
                    "unknown_level",
                    "vintage",
                    "population_role"
                ],
                dropna=False
            )
            .size()
            .reset_index(name="row_count")
        )

if unknown_category_results:
    unknown_category_audit = pd.concat(
        unknown_category_results,
        ignore_index=True
    )
else:
    unknown_category_audit = pd.DataFrame(
        columns=[
            "feature",
            "unknown_level",
            "vintage",
            "population_role",
            "row_count"
        ]
    )

display(unknown_category_audit)

print(
    "Unknown feature-level combinations:",
    len(unknown_category_audit)
)
print(
    "Total rows containing audited unknown levels:",
    unknown_category_audit["row_count"].sum()
)

,feature,unknown_level,vintage,population_role,row_count
0,channel,T,2006,historical_comparison,22031


Unknown feature-level combinations: 1
Total rows containing audited unknown levels: 22031


In [6]:
unknown_audit_path = (
    project_root
    / "data"
    / "processed"
    / "unknown_category_audit.csv"
)

unknown_category_audit.to_csv(
    unknown_audit_path,
    index=False
)

historical_eligible_loans = (
    prediction_output["vintage"]
    .eq(2006)
    .sum()
)

unknown_historical_rows = int(
    unknown_category_audit["row_count"].sum()
)

unknown_category_conclusion = pd.Series({
    "unknown_categories_in_2017_test": 0,
    "historical_2006_eligible_loans":
        historical_eligible_loans,
    "historical_2006_unknown_channel_rows":
        unknown_historical_rows,
    "historical_2006_unknown_percent": round(
        100
        * unknown_historical_rows
        / historical_eligible_loans,
        2
    ),
    "primary_2017_test_affected": False,
    "historical_comparison_requires_limitation":
        True,
    "audit_file_exists":
        unknown_audit_path.exists()
})

display(unknown_category_conclusion)

unknown_categories_in_2017_test                  0
historical_2006_eligible_loans               37555
historical_2006_unknown_channel_rows         22031
historical_2006_unknown_percent              58.66
primary_2017_test_affected                   False
historical_comparison_requires_limitation     True
audit_file_exists                             True
dtype: object

In [7]:
decile_results = []

for population_name, population_data in (
    prediction_output.groupby("population_role")
):
    scored_population = population_data.copy()

    # Decile 10 represents the highest predicted risk.
    scored_population["risk_decile"] = (
        pd.qcut(
            scored_population[
                "predicted_default_probability"
            ].rank(method="first"),
            q=10,
            labels=False
        )
        + 1
    )

    overall_default_rate = (
        scored_population["default_24m"].mean()
    )

    population_deciles = (
        scored_population
        .groupby("risk_decile", observed=False)
        .agg(
            loans=("loan_identifier", "size"),
            defaults=("default_24m", "sum"),
            mean_predicted_probability=(
                "predicted_default_probability",
                "mean"
            )
        )
        .reset_index()
    )

    population_deciles[
        "observed_default_rate"
    ] = (
        population_deciles["defaults"]
        / population_deciles["loans"]
    )

    population_deciles["lift"] = (
        population_deciles[
            "observed_default_rate"
        ]
        / overall_default_rate
    )

    population_deciles["default_capture"] = (
        population_deciles["defaults"]
        / scored_population["default_24m"].sum()
    )

    population_deciles.insert(
        0,
        "population_role",
        population_name
    )

    decile_results.append(population_deciles)

decile_analysis = pd.concat(
    decile_results,
    ignore_index=True
)

decile_path = (
    project_root
    / "data"
    / "processed"
    / "model_decile_analysis.csv"
)

decile_analysis.to_csv(
    decile_path,
    index=False
)

final_test_deciles = (
    decile_analysis.loc[
        decile_analysis[
            "population_role"
        ].eq("later_vintage_test")
    ]
    .sort_values(
        "risk_decile",
        ascending=False
    )
    .copy()
)

for column in [
    "mean_predicted_probability",
    "observed_default_rate",
    "default_capture"
]:
    final_test_deciles[column] = (
        100 * final_test_deciles[column]
    ).round(3)

final_test_deciles["lift"] = (
    final_test_deciles["lift"].round(2)
)

display(final_test_deciles)

print("Decile file saved:", decile_path.exists())

,population_role,risk_decile,loans,defaults,mean_predicted_probability,observed_default_rate,lift,default_capture
19,later_vintage_test,10,4207,200,4.375,4.754,5.04,50.378
18,later_vintage_test,9,4206,76,1.507,1.807,1.91,19.144
17,later_vintage_test,8,4206,50,0.929,1.189,1.26,12.594
16,later_vintage_test,7,4207,27,0.637,0.642,0.68,6.801
15,later_vintage_test,6,4206,22,0.456,0.523,0.55,5.542
14,later_vintage_test,5,4206,9,0.330,0.214,0.23,2.267
13,later_vintage_test,4,4207,4,0.236,0.095,0.1,1.008
12,later_vintage_test,3,4206,6,0.163,0.143,0.15,1.511
11,later_vintage_test,2,4206,1,0.104,0.024,0.03,0.252
10,later_vintage_test,1,4207,2,0.050,0.048,0.05,0.504


Decile file saved: True


In [8]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)

performance_results = []

for population_name, population_data in (
    prediction_output.groupby("population_role")
):
    actual = (
        population_data["default_24m"]
        .astype(int)
        .to_numpy()
    )

    predicted = population_data[
        "predicted_default_probability"
    ].to_numpy()

    top_decile = decile_analysis.loc[
        decile_analysis[
            "population_role"
        ].eq(population_name)
        & decile_analysis["risk_decile"].eq(10)
    ].iloc[0]

    observed_rate = actual.mean()
    predicted_rate = predicted.mean()

    performance_results.append({
        "population_role": population_name,
        "rows": len(population_data),
        "defaults": int(actual.sum()),
        "observed_default_rate": observed_rate,
        "mean_predicted_probability": predicted_rate,
        "prediction_to_observed_ratio": (
            predicted_rate / observed_rate
        ),
        "roc_auc": roc_auc_score(
            actual,
            predicted
        ),
        "average_precision": average_precision_score(
            actual,
            predicted
        ),
        "brier_score": brier_score_loss(
            actual,
            predicted
        ),
        "top_decile_lift": top_decile["lift"],
        "top_decile_default_capture": (
            top_decile["default_capture"]
        )
    })

model_performance_summary = pd.DataFrame(
    performance_results
)

performance_path = (
    project_root
    / "data"
    / "processed"
    / "model_performance_summary.csv"
)

model_performance_summary.to_csv(
    performance_path,
    index=False
)

display(
    model_performance_summary.round(4)
)

print(
    "Performance summary saved:",
    performance_path.exists()
)

,population_role,rows,defaults,observed_default_rate,mean_predicted_probability,prediction_to_observed_ratio,roc_auc,average_precision,brier_score,top_decile_lift,top_decile_default_capture
0,historical_comparison,37555,967,0.0257,0.0325,1.2641,0.8018,0.0915,0.0253,3.8154,0.3816
1,later_vintage_test,42064,397,0.0094,0.0088,0.9311,0.8403,0.0591,0.0091,5.0371,0.5038
2,model_development,82387,583,0.0071,0.0071,1.0094,0.8571,0.0479,0.0069,5.6430,0.5643


Performance summary saved: True


In [9]:
import numpy as np

transformed_feature_names = (
    reloaded_pd_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

model_coefficients = (
    reloaded_pd_pipeline
    .named_steps["model"]
    .coef_[0]
)

coefficient_analysis = pd.DataFrame({
    "transformed_feature":
        transformed_feature_names,
    "coefficient":
        model_coefficients,
    "odds_ratio":
        np.exp(model_coefficients),
    "absolute_coefficient":
        np.abs(model_coefficients)
})

coefficient_analysis = (
    coefficient_analysis
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

coefficient_path = (
    project_root
    / "data"
    / "processed"
    / "model_coefficients.csv"
)

coefficient_analysis.to_csv(
    coefficient_path,
    index=False
)

display(
    coefficient_analysis.head(15).round(4)
)

print(
    "Coefficient file saved:",
    coefficient_path.exists()
)
print(
    "Transformed coefficients:",
    len(coefficient_analysis)
)

,transformed_feature,coefficient,odds_ratio,absolute_coefficient
0,categorical__property_state_FL,1.3604,3.8978,1.3604
1,categorical__property_state_KY,-1.0726,0.3421,1.0726
2,categorical__property_state_NH,-0.9339,0.3930,0.9339
3,categorical__property_state_LA,0.9324,2.5407,0.9324
4,categorical__property_state_RI,-0.9007,0.4063,0.9007
5,categorical__property_state_HI,0.7988,2.2228,0.7988
6,categorical__property_state_OR,-0.7597,0.4678,0.7597
7,categorical__property_state_TX,0.7353,2.0862,0.7353
8,numeric__credit_score,-0.7232,0.4852,0.7232
9,categorical__property_state_VT,0.6880,1.9898,0.6880


Coefficient file saved: True
Transformed coefficients: 84


In [10]:
numeric_coefficient_review = (
    coefficient_analysis.loc[
        coefficient_analysis[
            "transformed_feature"
        ].str.startswith("numeric__")
    ]
    .copy()
)

numeric_coefficient_review[
    "feature"
] = (
    numeric_coefficient_review[
        "transformed_feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
)

numeric_coefficient_review = (
    numeric_coefficient_review[
        [
            "feature",
            "coefficient",
            "odds_ratio",
            "absolute_coefficient"
        ]
    ]
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    numeric_coefficient_review.round(4)
)

,feature,coefficient,odds_ratio,absolute_coefficient
0,credit_score,-0.7232,0.4852,0.7232
1,number_of_borrowers,-0.4368,0.6461,0.4368
2,original_cltv_capped_200,0.2542,1.2895,0.2542
3,original_dti,0.2528,1.2877,0.2528
4,original_interest_rate,0.1598,1.1733,0.1598
5,original_dti_missing,0.1193,1.1267,0.1193
6,mi_percentage,0.1066,1.1125,0.1066
7,original_loan_term,0.0728,1.0755,0.0728
8,original_upb_log,-0.0715,0.9310,0.0715
9,original_ltv_capped_200,0.0366,1.0373,0.0366
